In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn import metrics
import seaborn as sns
import numpy as np
from pathlib import Path
import os

In [ ]:
data_pipeline_name = "features-v1"

In [ ]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../../"

In [ ]:
experiment_path = Path(output_path) / "data" / f"{data_pipeline_name}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [ ]:
ss = pd.read_csv("../../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [ ]:
X = pd.read_csv("../../../data/raw/train.csv")
X_test = pd.read_csv("../../../data/raw/test.csv")
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  str    
 11  stress_level             636221 non-null  str    
 12  academic_work_impact     647145 non-null  str    
 13  addicted_label           691369 non-null  int64  
dtypes: float64(9), 

In [20]:
cat_cols = ["gender"]
ordinal_cols = ["stress_level"]
binary_cols = ["academic_work_impact"]

In [ ]:
for frame in [X, X_test]:
    frame.drop('id', axis=1, inplace=True)

    for col in cat_cols:
        frame[col] = frame[col].astype('category')

    frame['stress_level'] = frame['stress_level'].replace({'Low':0, 'Medium':1, 'High':2})
    frame['academic_work_impact'] = frame['academic_work_impact'].replace({'No':0, 'Yes':1})

    for col in ordinal_cols + binary_cols:
        frame[col] = frame[col].astype(np.float64)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   age                      662440 non-null  float64 
 1   daily_screen_time_hours  595515 non-null  float64 
 2   social_media_hours       557374 non-null  float64 
 3   gaming_hours             564548 non-null  float64 
 4   work_study_hours         639851 non-null  float64 
 5   sleep_hours              646889 non-null  float64 
 6   notifications_per_day    623785 non-null  float64 
 7   app_opens_per_day        610659 non-null  float64 
 8   weekend_screen_time      579306 non-null  float64 
 9   gender                   662335 non-null  category
 10  stress_level             636221 non-null  float64 
 11  academic_work_impact     647145 non-null  float64 
 12  addicted_label           691369 non-null  int64   
dtypes: category(1), float64(11), int64(1)
memory usage: 64.

In [22]:
for frame in [X, X_test]:
    frame['screen_sleep_ratio'] = frame['daily_screen_time_hours'] / frame['sleep_hours']
    frame['total_activity'] = (frame['social_media_hours'] + frame['gaming_hours'] + frame['work_study_hours'])

    frame['social_media_ratio'] = frame['social_media_hours'] / (frame['total_activity'] + 1)
    frame['gaming_ratio'] = frame['gaming_hours'] / (frame['total_activity'] + 1)
    frame['work_ratio'] = frame['work_study_hours'] / (frame['total_activity'] + 1)

    frame['daily_free_hours'] = 24 - (frame['daily_screen_time_hours'] + frame['sleep_hours'])
    frame["daily_extra_screen_time_hours"] = frame['daily_screen_time_hours'] - (frame['social_media_hours'] + frame['gaming_hours'] + frame['work_study_hours'])
    frame['weekend_extra_screen_time'] = frame['weekend_screen_time'] - frame['daily_screen_time_hours']
    frame['phone_activity'] = frame['notifications_per_day'] * frame['app_opens_per_day']
    frame['hourly_notifications_per_day'] = frame['notifications_per_day'] / frame['daily_screen_time_hours']
    frame['hourly_app_opens_per_day'] = frame['app_opens_per_day'] / frame['daily_screen_time_hours']

    frame['stress_daily_hours_impact'] = (frame['stress_level'] * frame['daily_screen_time_hours']).astype(np.float64)
    frame['stress_social_media_hours_impact'] = (frame['stress_level'] * frame['social_media_hours']).astype(np.float64)
    frame['stress_work_study_hours_impact'] = (frame['stress_level'] * frame['work_study_hours']).astype(np.float64)

    frame['stress_app_opens_impact'] =  (frame['stress_level'] * frame['app_opens_per_day']).astype(np.float64)
    frame['stress_notifications_impact'] =  (frame['stress_level'] * frame['notifications_per_day']).astype(np.float64)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 29 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              647145 non-null 

In [23]:
y = X[target_column]
X = X.drop(target_column, axis=1)

In [ ]:
X.to_csv('../../../data/processed/train_features.csv', index=False)
y.to_csv('../../../data/processed/train_labels.csv', index=False)

X_test.to_csv('../../../data/processed/test_features.csv', index=False)

In [25]:
for ft in cat_cols:
    X = pd.concat([X, pd.get_dummies(X[ft])], axis=1)
    X.drop(ft, axis=1, inplace=True)

    X_test = pd.concat([X_test, pd.get_dummies(X_test[ft])], axis=1)
    X_test.drop(ft, axis=1, inplace=True)

In [ ]:
X.to_csv('../../../data/processed/train_features_ohe.csv', index=False)
X_test.to_csv('../../../data/processed/test_features_ohe.csv', index=False)

In [27]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 30 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               662440 non-null  float64
 1   daily_screen_time_hours           595515 non-null  float64
 2   social_media_hours                557374 non-null  float64
 3   gaming_hours                      564548 non-null  float64
 4   work_study_hours                  639851 non-null  float64
 5   sleep_hours                       646889 non-null  float64
 6   notifications_per_day             623785 non-null  float64
 7   app_opens_per_day                 610659 non-null  float64
 8   weekend_screen_time               579306 non-null  float64
 9   stress_level                      636221 non-null  float64
 10  academic_work_impact              647145 non-null  float64
 11  screen_sleep_ratio                559350 non-null  float64
 12 

In [28]:
binary_cols = binary_cols + (X.select_dtypes(include="bool").columns.to_list())
binary_cols

['academic_work_impact', 'Female', 'Male', 'Other']

In [29]:
X_cv_imputed = np.empty((X.shape[0], X.shape[1]), dtype=float)

kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    
    imp = SimpleImputer(strategy='median')
    
    X_train_imp = imp.fit_transform(X_train)
    X_valid_imp = imp.transform(X_valid)
    
    X_cv_imputed[valid_index] = X_valid_imp

X_cv_imputed = pd.DataFrame(X_cv_imputed, columns=X.columns, index=X.index)

imp = SimpleImputer(strategy='median')
imp.fit(X)
X_test_imp = imp.transform(X_test)
X_test_cv_imputed = pd.DataFrame(X_test_imp, columns=X_test.columns, index=X_test.index)

X = X_cv_imputed.copy()
X_test = X_test_cv_imputed.copy()

for col in binary_cols:
    X[col] = X[col].round().clip(0, 1).astype(bool)
    X_test[col] = X_test[col].round().clip(0, 1).astype(bool)

In [ ]:
X.to_csv('../../../data/processed/train_features_ohe_imputed.csv', index=False)
X_test.to_csv('../../../data/processed/test_features_ohe_imputed.csv', index=False)

In [31]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 30 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               691369 non-null  float64
 1   daily_screen_time_hours           691369 non-null  float64
 2   social_media_hours                691369 non-null  float64
 3   gaming_hours                      691369 non-null  float64
 4   work_study_hours                  691369 non-null  float64
 5   sleep_hours                       691369 non-null  float64
 6   notifications_per_day             691369 non-null  float64
 7   app_opens_per_day                 691369 non-null  float64
 8   weekend_screen_time               691369 non-null  float64
 9   stress_level                      691369 non-null  float64
 10  academic_work_impact              691369 non-null  bool   
 11  screen_sleep_ratio                691369 non-null  float64
 12 

In [32]:
log_cols = ["social_media_hours", "gaming_hours", "work_study_hours", "daily_extra_screen_time_hours", "phone_activity", "hourly_notifications_per_day", "hourly_app_opens_per_day"]
std_cols = ["total_activity", "work_ratio", "gaming_ratio", "social_media_ratio", "screen_sleep_ratio", "daily_screen_time_hours", "weekend_screen_time", "daily_free_hours", "weekend_extra_screen_time", "stress_daily_hours_impact", "stress_social_media_hours_impact", "stress_work_study_hours_impact"]
minmax_cols = ["age", "sleep_hours", "notifications_per_day", "app_opens_per_day", "stress_level", "stress_app_opens_impact", "stress_notifications_impact"]
len_norm_cols = len(log_cols) + len(std_cols) + len(minmax_cols)
len_norm_cols

26

In [33]:
for col in binary_cols:
    X[col] = X[col].astype(int)
    X_test[col] = X_test[col].astype(int)

In [34]:
for col in log_cols:
    X[col] = np.log1p(X[col])
    X_test[col] = np.log1p(X_test[col])

In [35]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 30 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               691369 non-null  float64
 1   daily_screen_time_hours           691369 non-null  float64
 2   social_media_hours                691369 non-null  float64
 3   gaming_hours                      691369 non-null  float64
 4   work_study_hours                  691369 non-null  float64
 5   sleep_hours                       691369 non-null  float64
 6   notifications_per_day             691369 non-null  float64
 7   app_opens_per_day                 691369 non-null  float64
 8   weekend_screen_time               691369 non-null  float64
 9   stress_level                      691369 non-null  float64
 10  academic_work_impact              691369 non-null  int64  
 11  screen_sleep_ratio                691369 non-null  float64
 12 

In [36]:
scale_cols = std_cols + log_cols

X_cv_scaled = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)

kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train[scale_cols])
    X_valid_scaled = scaler.transform(X_valid[scale_cols])
    
    X_valid_full = X_valid.copy()
    X_valid_full[scale_cols] = X_valid_scaled
    
    X_cv_scaled.loc[valid_index] = X_valid_full

scaler = StandardScaler()
scaler.fit(X[scale_cols])

X_test_scaled = scaler.transform(X_test[scale_cols])

X_test_full = X_test.copy()
X_test_full[scale_cols] = X_test_scaled

X = X_cv_scaled.copy()
X_test = X_test_full.copy()

In [37]:
X_cv_scaled = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)

kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train[minmax_cols])
    X_valid_scaled = scaler.transform(X_valid[minmax_cols])
    
    X_valid_full = X_valid.copy()
    X_valid_full[minmax_cols] = X_valid_scaled
    
    X_cv_scaled.loc[valid_index] = X_valid_full

scaler = MinMaxScaler()
scaler.fit(X[minmax_cols])

X_test_scaled = scaler.transform(X_test[minmax_cols])

X_test_full = X_test.copy()
X_test_full[minmax_cols] = X_test_scaled

X = X_cv_scaled.copy()
X_test = X_test_full.copy()

In [ ]:
X.to_csv('../../../data/processed/train_features_ohe_imputed_scaled.csv', index=False)
X_test.to_csv('../../../data/processed/test_features_ohe_imputed_scaled.csv', index=False)

In [39]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 30 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               691369 non-null  float64
 1   daily_screen_time_hours           691369 non-null  float64
 2   social_media_hours                691369 non-null  float64
 3   gaming_hours                      691369 non-null  float64
 4   work_study_hours                  691369 non-null  float64
 5   sleep_hours                       691369 non-null  float64
 6   notifications_per_day             691369 non-null  float64
 7   app_opens_per_day                 691369 non-null  float64
 8   weekend_screen_time               691369 non-null  float64
 9   stress_level                      691369 non-null  float64
 10  academic_work_impact              691369 non-null  float64
 11  screen_sleep_ratio                691369 non-null  float64
 12 